# Part L — Fisher information in the 408 MHz prediction

The Fisher-matrix numbers of the paper's Section 6.3 (`fig:tris-anchor` paragraph, main.tex:625): how much of the
information on ln A at 408 MHz each data set carries, and how σ(ln A) grows when data sets are left out. Ported from
the old notebook 08 (`08_update_plots.ipynb`, cell 54, "Part 2 — the information share in the 408 MHz prediction"),
with its definitions unchanged:

* **Jacobian** of T = A (ν/408)^β with respect to (ln A, β) at each ring sample's power-law fit:
  ∂T/∂ln A = T, ∂T/∂β = T ln(ν/408).
* **Fisher matrix** F = Jᵀ diag(1/σ²) J with the per-point errors as they are (zero-level terms inside, no offsets).
  σ(ln A at 408 MHz) = √(F⁻¹)₁₁, β free.
* **Leave-out:** the same Jacobian with data sets dropped: all, each one left out, both TRIS bands left out.
* **Information share at fixed β:** each data set's (T_model/σ)² as a fraction of the sum.
* **Samples:** Cygnus A masked (RA 299.9 ± 25°), every data set and Haslam finite: 103 of 120. Medians over them.

| | |
|---|---|
| **Reads** | `paper_outputs/pipeline/step2_ring_tods/ring_tods.npz` (TRIS ring data, beam-matched survey TODs, errors), `paper_outputs/pipeline/step5_fit/` (our 13-point power-law fit), `configs/run_pipeline.yaml` |
| **Writes** | `paper_outputs/fisher_share/` (`fisher_sigma_lnA_leave_out.csv`, `fisher_share.csv`, both for the 13-point fit) |
| **Feeds the paper** | main.tex:625 (Fisher shares and σ(ln A)), and g at the median index (main.tex:625–626) |

**Code reused, not copied:** the data are step 2's table (`pipeline.load_ring_tods`); the 13-point fit is step 5's
(`pipeline.our_model`, UltraNest's maximum-likelihood point); the six-survey check fits with
`sed_prototype.fit_power_law` (minimum χ², as notebook 08's `inner_fit`); Haslam = g P + z uses `pipeline.york`. Only
the Fisher matrix and the shares are new code here.

In [1]:
%matplotlib inline
import time as _nb_time; _NB_T0 = _nb_time.time()   # notebook run time, printed by the last cell
import sys
from pathlib import Path

import numpy as np
import pandas as pd

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs" / "run_pipeline.yaml").exists())
sys.path.insert(0, str(REPO / "src"))
from moonrunes import pipeline
from moonrunes.config import load_config
from moonrunes.sed_prototype import fit_power_law, sigma_for

config = load_config(REPO / "configs" / "run_pipeline.yaml")
OUT_DIR = REPO / "paper_outputs" / "fisher_share"
OUT_DIR.mkdir(parents=True, exist_ok=True)

NU0 = 408.0                         # the pivot: A is the temperature at 408 MHz
CYG_RA, CYG_HALF = 299.9, 25.0      # Cygnus A mask, as notebook 08
SIX = ["LWA_35MHz.fits", "MAIPU_45mhZ.fits", "LWA_80MHz.fits", "TRIS_600", "TRIS_820",
       "STOCKERT_VILLA_1400MHZ.fits"]                       # notebook 08's six ring surveys (the check only)
# notebook 08's printed values (cell 54, run 2026-11-04), for the check on the six surveys
NB08_SIGMA = {"all": 0.00591, "without LWA1 35": 0.00591, "without Maipu 45": 0.02347, "without LWA1 80": 0.00591,
              "without TRIS 600": 0.06528, "without TRIS 820": 0.00592, "without Stockert 1420": 0.00591,
              "without TRIS 600 and 820": 0.25207}
NB08_SHARE = {"LWA1 35": (0.0009, 0.0003, 0.0014), "Maipu 45": (0.1091, 0.0449, 0.2070), "LWA1 80": (0.0007, 0.0002, 0.0011),
              "TRIS 600": (0.8845, 0.7867, 0.9493), "TRIS 820": (0.0045, 0.0040, 0.0051), "Stockert 1420": (0.0002, 0.0002, 0.0002)}
NB08_G = {-2.4839: 1.0394}         # notebook 08 Part 3: g at the six-survey median beta

### Data and samples

Step 2's SED table: TRIS 600 and 820 ring data with `ring_sigma`, every other data set's beam-matched ring TOD with
its published errors, and Haslam's beam-matched TOD (`sigma_for`, 10% ⊕ 3 K). These are the same values notebook 08
read (TRIS from notebook 12's cache, the surveys and Haslam from notebook 18's beam-matched TODs).

In [2]:
D = pipeline.load_ring_tods(config)
RA, N = np.asarray(D["ra"], float), D["ra"].size
FIT_SETS = pipeline.fit_data_sets(config)                 # step 5's 13 data sets, low frequency first
KEYS = list(FIT_SETS)
assert set(SIX) <= set(KEYS), KEYS
TH, SH = np.asarray(D["haslam_T"], float), np.asarray(D["haslam_S"], float)
assert np.allclose(SH[np.isfinite(TH)], np.asarray(sigma_for(dict(freq_mhz=NU0, file=D["target_file"]), TH[np.isfinite(TH)]), float))

d_cyg = np.abs((RA - CYG_RA + 180.0) % 360.0 - 180.0)
KEEP = (d_cyg > CYG_HALF) & np.all([np.isfinite(D["T"][k]) for k in KEYS], axis=0) & np.isfinite(TH)
IDX = np.flatnonzero(KEEP)
NS = IDX.size

def table(keys):
    """(T, sigma, x = ln(nu/408)) over the kept samples, columns in the order of keys."""
    T = np.stack([D["T"][k][IDX] for k in keys], axis=1)
    S = np.stack([D["S"][k][IDX] for k in keys], axis=1)
    return T, S, np.log(np.array([D["nu"][k] for k in keys]) / NU0)

print(f"{NS} of {N} ring samples (Cygnus A RA {CYG_RA - CYG_HALF:g}–{CYG_RA + CYG_HALF:g}° masked; every data set and Haslam finite)")
print("data sets (step 5):", ", ".join(f"{FIT_SETS[k]} ({D['nu'][k]:g} MHz)" for k in KEYS))

103 of 120 ring samples (Cygnus A RA 274.9–324.9° masked; every data set and Haslam finite)
data sets (step 5): LWA1 35 (35 MHz), LWA1 38 (38 MHz), LWA1 40 (40 MHz), LWA1 45 (45 MHz), Maipu 45 (45 MHz), LWA1 50 (50 MHz), LWA1 60 (60 MHz), LWA1 70 (70 MHz), LWA1 74 (74 MHz), LWA1 80 (80 MHz), TRIS 600 (600.5 MHz), TRIS 820 (817.8 MHz), Stockert 1420 (1420 MHz)


### The Fisher computation (ported from notebook 08, cell 54)

`fisher(T_model, S, x)` takes the model temperatures at the data points (the power law at a fit), the errors and
x = ln(ν/408), and returns, for any subset of columns, σ(ln A) per sample (β free) and each column's information share
at fixed β.

In [3]:
def fisher(TM, S, x):
    """Notebook 08's Part 2: J = [dT/dlnA, dT/dbeta] = [T, T x]; F = J^T diag(1/s^2) J per sample."""
    J = np.stack([TM, TM * x], axis=2)                    # (samples, points, 2)
    W = 1.0 / S ** 2
    def sigma_lnA(cols):
        F = np.einsum("sni,sn,snj->sij", J[:, cols, :], W[:, cols], J[:, cols, :])
        return np.sqrt(np.linalg.inv(F)[:, 0, 0])
    info = TM ** 2 * W                                    # (dT/dlnA)^2 / sigma^2 at fixed beta
    return sigma_lnA, info / info.sum(axis=1, keepdims=True)

def leave_out_table(keys, sigma_lnA):
    """Median sigma(ln A_408): all, each data set left out, both TRIS bands left out (notebook 08's rows)."""
    allc = list(range(len(keys)))
    sig = {"all": sigma_lnA(allc)}
    for j, k in enumerate(keys):
        sig[f"without {FIT_SETS[k]}"] = sigma_lnA([i for i in allc if i != j])
    tris = [keys.index("TRIS_600"), keys.index("TRIS_820")]
    sig["without TRIS 600 and 820"] = sigma_lnA([i for i in allc if i not in tris])
    base = np.median(sig["all"])
    return sig, pd.DataFrame([{"data sets used": lab, "median σ(ln A_408)": np.median(v),
                               "median σ(ln A_408) − all": np.median(v) - base} for lab, v in sig.items()])

def share_table(keys, share):
    return pd.DataFrame([{"data set": FIT_SETS[k], "median share (β fixed)": np.median(share[:, j]),
                          "min": share[:, j].min(), "max": share[:, j].max()} for j, k in enumerate(keys)])

def haslam_g(beta):
    """Notebook 08 Part 3: TRIS-only P(408) with a fixed extrapolation index (weighted mean of the two TRIS points
    taken to 408 MHz with beta), then Haslam = g P + z by York (pipeline.york, sigma inflated when chi2/nu > 1)."""
    t6, t8 = D["T"]["TRIS_600"][IDX], D["T"]["TRIS_820"][IDX]
    s6, s8 = D["S"]["TRIS_600"][IDX], D["S"]["TRIS_820"][IDX]
    f6, f8 = (NU0 / D["nu"]["TRIS_600"]) ** beta, (NU0 / D["nu"]["TRIS_820"]) ** beta
    w6, w8 = 1.0 / (s6 * f6) ** 2, 1.0 / (s8 * f8) ** 2
    P, SP = (w6 * t6 * f6 + w8 * t8 * f8) / (w6 + w8), 1.0 / np.sqrt(w6 + w8)
    f = pipeline.york(P, TH[IDX], SP, SH[IDX])
    return dict(g=f["a"], sg=f["sa"], z=f["b"], sz=f["sb"])

### Check: the six surveys of notebook 08

The ported calculation on notebook 08's setup — its six surveys, a minimum-χ² power law at each sample
(`sed_prototype.fit_power_law`, standing in for notebook 08's `inner_fit`) — against notebook 08's printed values, at
the precision it printed them (5 decimals for σ, 4 for the shares). The fit's own σ(ln A) must equal the Fisher σ with
all six (notebook 08's assertion).

In [4]:
T6, S6, X6 = table(SIX)
P6 = np.array([[np.log(f["A"]), f["beta"], f["sigma_A"] / f["A"]] for f in
               (fit_power_law(np.exp(X6) * NU0, T6[i], S6[i]) for i in range(NS))])
TM6 = np.exp(P6[:, 0:1] + P6[:, 1:2] * X6)
sig6_fn, share6 = fisher(TM6, S6, X6)
SIG6, tab6 = leave_out_table(SIX, sig6_fn)
assert np.allclose(SIG6["all"], P6[:, 2], rtol=1e-5), "Fisher σ(ln A) is not the fit's own σ(ln A)"
s6tab = share_table(SIX, share6)

ok_sig = all(round(float(np.median(SIG6[lab])), 5) == v for lab, v in NB08_SIGMA.items())
ok_share = all(tuple(round(float(x), 4) for x in s6tab.set_index("data set").loc[n].to_numpy()) == v for n, v in NB08_SHARE.items())
beta6 = float(np.median(P6[:, 1]))
g6 = haslam_g(beta6)
ok_g = round(beta6, 4) in NB08_G and round(g6["g"], 4) == NB08_G[round(beta6, 4)]
print(tab6.to_string(index=False, float_format=lambda v: f"{v:.5f}"))
print()
print(s6tab.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
print(f"\nsix-survey median β {beta6:+.4f}; g at it {g6['g']:.4f} (notebook 08: β −2.4839, g 1.0394)")
print(f"CHECK against notebook 08: σ(ln A) {'REPRODUCED' if ok_sig else 'NOT REPRODUCED'}; "
      f"shares {'REPRODUCED' if ok_share else 'NOT REPRODUCED'}; g at the median β {'REPRODUCED' if ok_g else 'NOT REPRODUCED'}")
CHECK_PASSED = ok_sig and ok_share and ok_g

          data sets used  median σ(ln A_408)  median σ(ln A_408) − all
                     all             0.00591                   0.00000
         without LWA1 35             0.00591                   0.00000
        without Maipu 45             0.02347                   0.01757
         without LWA1 80             0.00591                   0.00000
        without TRIS 600             0.06528                   0.05937
        without TRIS 820             0.00592                   0.00001
   without Stockert 1420             0.00591                   0.00000
without TRIS 600 and 820             0.25207                   0.24616

     data set  median share (β fixed)    min    max
      LWA1 35                  0.0009 0.0003 0.0014
     Maipu 45                  0.1091 0.0449 0.2070
      LWA1 80                  0.0007 0.0002 0.0011
     TRIS 600                  0.8845 0.7867 0.9493
     TRIS 820                  0.0045 0.0040 0.0051
Stockert 1420                  0.0002 0.0002 0.0

### The current fit: our 13-point power law (pipeline step 5)

The Jacobian is taken at step 5's fit at each sample (UltraNest's maximum-likelihood point of the power law, the model
step 4 selected), with step 2's errors: all 13 data sets, TRIS included.

In [5]:
model, fn, FITS5 = pipeline.our_model(config)
assert model == "power law", model
T13, S13, X13 = table(KEYS)
for i, u in enumerate(IDX):                                # step 5 fitted exactly these points
    r = FITS5[int(u)]
    assert [str(k) for k in r["names"]] == ["A_K", "beta"]
    assert np.allclose(np.array([r["t"][list(map(str, r["keys"])).index(k)] for k in KEYS]), T13[i])
PAR = np.array([np.asarray(FITS5[int(u)]["maxpoint"], float) for u in IDX])           # (A, beta)
TM13 = PAR[:, 0:1] * np.exp(PAR[:, 1:2] * X13)
sig13_fn, share13 = fisher(TM13, S13, X13)
SIG13, tab13 = leave_out_table(KEYS, sig13_fn)
s13tab = share_table(KEYS, share13)
post_sd = np.array([np.std(np.log(FITS5[int(u)]["samples"][:, 0])) for u in IDX])    # UltraNest's own σ(ln A)
tab13.to_csv(OUT_DIR / "fisher_sigma_lnA_leave_out.csv", index=False)
s13tab.to_csv(OUT_DIR / "fisher_share.csv", index=False)
with pd.option_context("display.width", 200):
    print(f"σ(ln A at 408 MHz), Fisher (ln A, β) at step 5's fit; median over {NS} samples\n")
    print(tab13.to_string(index=False, float_format=lambda v: f"{v:.6f}"))
    print("\nshare of the Fisher information on ln A at fixed β (median, min, max over samples)\n")
    print(s13tab.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
print(f"\nFisher σ(ln A), all: median {np.median(SIG13['all']):.5f}; UltraNest posterior sd of ln A: median "
      f"{np.median(post_sd):.5f} (ratio {np.median(post_sd / SIG13['all']):.3f})")
print("written:", ", ".join(str(p.relative_to(REPO)) for p in (OUT_DIR / "fisher_sigma_lnA_leave_out.csv", OUT_DIR / "fisher_share.csv")))

σ(ln A at 408 MHz), Fisher (ln A, β) at step 5's fit; median over 103 samples

          data sets used  median σ(ln A_408)  median σ(ln A_408) − all
                     all            0.005875                  0.000000
         without LWA1 35            0.005879                  0.000004
         without LWA1 38            0.005879                  0.000004
         without LWA1 40            0.005879                  0.000004
         without LWA1 45            0.005879                  0.000004
        without Maipu 45            0.012366                  0.006491
         without LWA1 50            0.005879                  0.000004
         without LWA1 60            0.005878                  0.000003
         without LWA1 70            0.005878                  0.000003
         without LWA1 74            0.005878                  0.000003
         without LWA1 80            0.005878                  0.000003
        without TRIS 600            0.064762                  0.05888

### The numbers main.tex:625 quotes, recomputed

Each line names the definition (notebook 08's) and prints the six-survey value (the paper's) next to the 13-point one.

In [6]:
def med(sig, lab): return float(np.median(sig[lab]))
def sh(tab, name, col="median share (β fixed)"): return float(tab.set_index("data set").loc[name, col])
others6 = [f"without {FIT_SETS[k]}" for k in SIX if k not in ("TRIS_600", "MAIPU_45mhZ.fits")]
others13 = [f"without {FIT_SETS[k]}" for k in KEYS if k not in ("TRIS_600", "MAIPU_45mhZ.fits")]
rows = [
 ("88 per cent", "TRIS 600's median share of the information on ln A at fixed β", sh(s6tab, "TRIS 600"), sh(s13tab, "TRIS 600")),
 ("79 per cent", "TRIS 600's share, minimum over samples", sh(s6tab, "TRIS 600", "min"), sh(s13tab, "TRIS 600", "min")),
 ("95 per cent", "TRIS 600's share, maximum over samples", sh(s6tab, "TRIS 600", "max"), sh(s13tab, "TRIS 600", "max")),
 ("11 per cent", "Maipu 45's median share at fixed β", sh(s6tab, "Maipu 45"), sh(s13tab, "Maipu 45")),
 ("0.006", "median σ(ln A_408), all data sets, β free", med(SIG6, "all"), med(SIG13, "all")),
 ("0.065", "median σ(ln A_408) without TRIS 600", med(SIG6, "without TRIS 600"), med(SIG13, "without TRIS 600")),
 ("0.25", "median σ(ln A_408) without TRIS 600 and 820", med(SIG6, "without TRIS 600 and 820"), med(SIG13, "without TRIS 600 and 820")),
 ("2e-5", "largest change of median σ(ln A_408) when one data set other than TRIS 600 and Maipu is left out",
  max(med(SIG6, l) - med(SIG6, "all") for l in others6), max(med(SIG13, l) - med(SIG13, "all") for l in others13)),
]
Q = pd.DataFrame(rows, columns=["paper quotes", "definition (notebook 08)", "six surveys (check)", "13 points (now)"])
with pd.option_context("display.width", 250, "display.max_colwidth", 120):
    print(Q.to_string(index=False, float_format=lambda v: f"{v:.6g}"))
worst = max(others13, key=lambda l: med(SIG13, l) - med(SIG13, "all"))
print(f"\n13 points: the largest of those changes is {worst} (+{med(SIG13, worst) - med(SIG13, 'all'):.2e}); "
      f"Maipu 45 left out: +{med(SIG13, 'without Maipu 45') - med(SIG13, 'all'):.5f}")

paper quotes                                                                         definition (notebook 08)  six surveys (check)  13 points (now)
 88 per cent                                    TRIS 600's median share of the information on ln A at fixed β             0.884451         0.879319
 79 per cent                                                           TRIS 600's share, minimum over samples             0.786692         0.776856
 95 per cent                                                           TRIS 600's share, maximum over samples             0.949278         0.947115
 11 per cent                                                               Maipu 45's median share at fixed β             0.109051         0.107146
       0.006                                                        median σ(ln A_408), all data sets, β free           0.00590588       0.00587524
       0.065                                                              median σ(ln A_408) without TRIS 600   

### g at the median index

Notebook 08 Part 3's TRIS-only prediction with the extrapolation index fixed at the median β of the power law through
all the data sets over these samples (the open triangle of `fig:tris-anchor`, −2.487 in `10_paper_plots` cell 22),
then Haslam = g P + z.

In [7]:
beta13 = float(np.median(PAR[:, 1]))
for lab, b in ((f"median β of step 5's fit over these {NS} samples", beta13), ("the value the figure marks", -2.487)):
    f = haslam_g(b)
    print(f"{lab}: β = {b:+.4f} → g = {f['g']:.4f} ± {f['sg']:.4f}, z = {f['z']:+.3f} ± {f['sz']:.3f} K")

median β of step 5's fit over these 103 samples: β = -2.4873 → g = 1.0381 ± 0.0526, z = -4.233 ± 1.435 K
the value the figure marks: β = -2.4870 → g = 1.0382 ± 0.0526, z = -4.233 ± 1.435 K


In [8]:
# Total notebook run time: from the first code cell (which sets _NB_T0) to here.
import time as _nb_time
if "_NB_T0" in globals():
    _dt = _nb_time.time() - _NB_T0
    print(f"total notebook run time: {_dt:.0f} s ({int(_dt // 3600)}h {int(_dt % 3600 // 60):02d}m {int(_dt % 60):02d}s)")
else:
    print("total notebook run time: not recorded (the first code cell was not run)")

total notebook run time: 3 s (0h 00m 02s)
